# prep_03_age
## Ohio Dental Clinic — Site Selection Analysis

**Purpose:** Processes ACS 5-Year Estimates table B01001 (Sex by Age). Extracts total population and age group breakdowns (0–14, 15–17, 18–24, 25–64, 65+) for all 1,233 Ohio ZCTAs. Population within 30-minute drive is the primary market size dimension (30% weight).

| | |
|---|---|
| **Input** | `B01001_age_raw.csv` (ACS 2020–2024) |
| **Output** | `B01001_age_cleaned.csv` |
| **Records** | 1,233 Ohio ZCTAs |

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# PATHS
RAW_DATA_PATH = r"C:\Users\mosun\Downloads\oh_clinic_rw_files"
OUTPUT_PATH = r"../data/cleaned"

# LOAD 
print("Loading B01001_age_raw.csv...")
df = pd.read_csv(
    os.path.join(RAW_DATA_PATH, "ACSDT5Y2024.B01001-Data.csv"),
    header=0,       # row 1 = column codes → used as column names
    skiprows=[1],   # row 2 = human labels → skip it
    dtype=str,
    low_memory=False
)
print(f"Raw file: {len(df)} rows | {len(df.columns)} columns")

# Extract 5-digit ZIP from NAME column
df["zip"] = df["NAME"].str.extract(r"(\d{5})")
df = df.dropna(subset=["zip"])
print(f"After ZIP extraction: {len(df)} rows")

def to_num(col):
    """Convert column to numeric, replacing Census suppression codes with 0."""
    s = pd.to_numeric(df[col], errors="coerce")
    s = s.replace(-666666666, np.nan).replace(-999999999, np.nan)
    return s.fillna(0)

# Total population
df["total_population"] = to_num("B01001_001E")

# Age 0-14: under 5 + 5-9 + 10-14 (male + female)
df["age_0_14"] = (
    to_num("B01001_003E") + to_num("B01001_004E") + to_num("B01001_005E") +
    to_num("B01001_027E") + to_num("B01001_028E") + to_num("B01001_029E")
)

# Age 15-17 (male + female)
df["age_15_17"] = (
    to_num("B01001_006E") +
    to_num("B01001_030E")
)

# Age 18-24: 18-19 + 20 + 21 + 22-24 (male + female)
df["age_18_24"] = (
    to_num("B01001_007E") + to_num("B01001_008E") +
    to_num("B01001_009E") + to_num("B01001_010E") +
    to_num("B01001_031E") + to_num("B01001_032E") +
    to_num("B01001_033E") + to_num("B01001_034E")
)

# Age 25-64: 25-29 through 62-64 (male 011-019 + female 035-043)
df["age_25_64"] = sum(
    to_num(f"B01001_{str(i).zfill(3)}E")
    for i in list(range(11, 20)) + list(range(35, 44))
)

# Age 65+: 65-66 through 85+ (male 020-025 + female 044-049)
df["age_65_plus"] = sum(
    to_num(f"B01001_{str(i).zfill(3)}E")
    for i in list(range(20, 26)) + list(range(44, 50))
)

# Sanity check — groups should sum to total population
df["_group_sum"] = (
    df["age_0_14"] + df["age_15_17"] + df["age_18_24"] +
    df["age_25_64"] + df["age_65_plus"]
)
df["_diff"] = (df["total_population"] - df["_group_sum"]).abs()
bad = df[df["_diff"] > 50]
if len(bad) > 0:
    print(f"Warning: {len(bad)} ZIPs have age group sum differing from total by >50 people")
else:
    print(f"Sanity check passed — age groups sum matches total population for all ZIPs")

# Keep final columns
age_clean = df[[
    "zip", "total_population",
    "age_0_14", "age_15_17", "age_18_24", "age_25_64", "age_65_plus"
]].copy()

# SAVE 
age_clean.to_csv(os.path.join(OUTPUT_PATH, "B01001_age_cleaned.csv"), index=False)
print(f"\nSaved: B01001_age_cleaned.csv")
print(f"Rows: {len(age_clean)} | Columns: {list(age_clean.columns)}")
print(f"Total Ohio population: {age_clean['total_population'].sum():,.0f}")
print(f"\nSample:")
print(age_clean.head(5).to_string(index=False))

Loading B01001_age_raw.csv...
Raw file: 1233 rows | 101 columns
After ZIP extraction: 1233 rows
Sanity check passed — age groups sum matches total population for all ZIPs

Saved: B01001_age_cleaned.csv
Rows: 1233 | Columns: ['zip', 'total_population', 'age_0_14', 'age_15_17', 'age_18_24', 'age_25_64', 'age_65_plus']
Total Ohio population: 11,810,293

Sample:
  zip  total_population  age_0_14  age_15_17  age_18_24  age_25_64  age_65_plus
43001              2892       435         94        211       1693          459
43002              1205       178          0        118        612          297
43003              3723       625         93        202       2212          591
43004             31477      7068       1397       2142      17176         3694
43005               339        99         45          0        109           86
